# Modelagem preditiva
_Machine Learning_

---

## Sumário

1. **Importação de bibliotecas**
2. **Carregamento da base**
    - 2.1. Carregamento dos dataframes

## 1. Importação de bibliotecas

In [8]:
# Importação de pacotes e definição de parâmetros globais

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.dataset as ds
import warnings
import gc
import os
import time
import optuna
import joblib
import random

from sklearn.model_selection import train_test_split
from pathlib import Path

In [9]:
# Configurações para exibição de dados no Jupyter Notebook

# Configurar opção para exibir todas as linhas do Dataframe
pd.set_option('display.max_rows', None)

# Configurar para exibir o conteúdo completo das colunas
pd.set_option('display.max_colwidth', None)

# Configurar a supressão de mensagens de aviso durante a execução
warnings.filterwarnings('ignore')

# Configurar estilo dos gráficos do Seaborn
sns.set_style('whitegrid')

## 2. Carregamento da base

In [10]:
# Efetuando a limpeza da memória antes do carregamento dos dados

print(f'\nQuantidade de objetos removidos da memória: {gc.collect()}')


Quantidade de objetos removidos da memória: 8


In [12]:
# Caminho para o diretório de dados
DATA_DIR = Path('../data/refined')

# Carregando os dados
try:
    dataset = ds.dataset(DATA_DIR, format='parquet')
    table = dataset.to_table()
    df = table.to_pandas()
except Exception as e:
    print(f'Erro ao carregar os dados: {e}')

In [13]:
print('VOLUMETRIA')
print(f'Quantidade de linhas (registros):  {df.shape[0]:,}')
print(f'Quantidade de colunas (variáveis): {df.shape[1]:,}')

VOLUMETRIA
Quantidade de linhas (registros):  10,000
Quantidade de colunas (variáveis): 34


In [14]:
df.head(10)

,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,...,relationship_strength,engagement_index,inactivity_risk_flag,high_value_customer,high_balance_inactive,high_value_low_products,senior_high_balance,inactive_high_salary,inactive_multi_product,inactive_long_term_customer
0,15565714,601,France,Male,47,1,64430.06,2,0,1,...,7.5,4,0,0,0,0,0,0,0,0
1,15565806,532,France,Male,38,9,0.00,2,0,0,...,8.5,2,0,0,0,0,0,0,1,1
2,15565879,845,France,Female,28,9,0.00,2,1,1,...,11.5,5,0,0,0,0,0,0,0,0
3,15565891,709,France,Male,39,8,0.00,2,1,0,...,8.0,3,0,0,0,0,0,0,1,1
4,15565996,653,France,Male,44,8,0.00,2,1,1,...,11.0,5,0,0,0,0,0,0,0,0
5,15566111,596,France,Male,39,9,0.00,1,1,0,...,6.5,2,0,0,0,0,0,0,0,1
6,15566139,526,France,Female,37,5,53573.18,1,1,0,...,4.5,2,1,0,0,0,0,0,0,0
7,15566251,618,France,Female,37,5,96652.86,1,1,0,...,4.5,2,1,0,0,0,0,0,0,0
8,15566269,787,France,Male,25,5,0.00,2,1,0,...,6.5,3,0,0,0,0,0,0,1,0
9,15566295,761,France,Female,33,6,138053.79,2,1,0,...,7.0,3,1,1,1,0,0,0,1,0


## 3. Preparação dos dados

### 3.1. Exibição dos metadados

In [16]:
# Função para geração de um dataframe de metadados

def generate_metadata(dataframe):
    '''
    Gera um DataFrame contendo metadados das colunas do DataFrame fornecido.

    :param dataframe: DataFrame
        DataFrame para o qual os metadados serão gerados.
    :return: DataFrame
        DataFrame contendo os metadados.
    '''
    metadata = pd.DataFrame({
        'Variável': dataframe.columns,
        'Tipo': dataframe.dtypes,
        'Qtde de nulos': dataframe.isnull().sum(),
        '% de nulos': round((dataframe.isnull().sum() / len(dataframe)) * 100, 2),
        'Cardinalidade': dataframe.nunique(),
    }).sort_values(by='Qtde de nulos', ascending=False).reset_index(drop=True)

    return metadata

In [17]:
generate_metadata(df)

,Variável,Tipo,Qtde de nulos,% de nulos,Cardinalidade
0,CustomerId,int32,0,0.0,10000
1,CreditScore,int32,0,0.0,460
2,Geography,str,0,0.0,3
3,Gender,str,0,0.0,2
4,Age,int32,0,0.0,70
5,Tenure,int32,0,0.0,11
6,Balance,float64,0,0.0,6382
7,NumOfProducts,int32,0,0.0,4
8,HasCrCard,int32,0,0.0,2
9,IsActiveMember,int32,0,0.0,2
